# Exploratory Data Analysis (EDA) on Retail Sales Data
Welcome to this beginner-friendly EDA! In this notebook, we will explore the `ecommerce_customer_behavior_dataset.csv` dataset, uncover patterns, and derive actionable business insights. We will follow a systematic checklist to ensure we cover all important aspects of data analysis.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set aesthetic style of the plots
sns.set_style('whitegrid')

## 1. Load Dataset and Initial Inspection
**Checklist Item:** Load dataset and perform initial inspection (shape, column dtypes, null value check)

Let's load the data using pandas and take a look at its basic structure.

In [ ]:
# Load the dataset
df = pd.read_csv('ecommerce_customer_behavior_dataset.csv')

# 1. Shape of the dataset (Rows, Columns)
print(f"Dataset Shape: {df.shape[0]} rows and {df.shape[1]} columns\n")

# 2. Data Types of each column
print("Column Data Types:")
print(df.dtypes)
print("\n")

# 3. Null values check
print("Null Values Check:")
print(df.isnull().sum())
print("\n")

# Display the first 5 rows
df.head()

**Observations:**
- We successfully loaded the dataset. It has many rows and columns like `Order_ID`, `Age`, `Total_Amount`, etc.
- The `Date` column is currently loaded as an `object` (string). We will need to convert this to a datetime format for our time-series analysis later.
- We checked for null (missing) values to see if we need to clean the data before proceeding. (You can see the counts of missing values above).

## 2. Descriptive Statistics
**Checklist Item:** Descriptive statistics (mean, median, mode, standard deviation for all numerical columns)

Here we will calculate basic statistics to understand the distribution of our numerical data (like Age, Total_Amount, etc).

In [ ]:
# Using describe() to get mean, standard deviation, min, max, and percentiles (which includes median at 50%)
descriptive_stats = df.describe().T
display(descriptive_stats)

# We can also explicitly calculate mode for numerical columns
print("\nMode for numerical columns:")
print(df.select_dtypes(include='number').mode().iloc[0])

**Observations:**
- **Mean & Median:** Comparing the mean (average) and the 50% (median) tells us if our data is skewed. For example, if the mean `Total_Amount` is much higher than the median, it means a few very large purchases are pulling the average up.
- **Standard Deviation (std):** This shows how spread out the numbers are. A high std in `Total_Amount` indicates a wide variety of purchase sizes.
- **Mode:** Shows the most frequently occurring value in each column.

## 3. Time Series Analysis
**Checklist Item:** Plot monthly and quarterly sales trends using line charts

First, we will convert the `Date` column to actual datetime objects. Then we will group the sales (`Total_Amount`) by month and quarter to see how revenue changes over time.

In [ ]:
# Convert 'Date' column to datetime
df['Date'] = pd.to_datetime(df['Date'])

# Create new columns for Month and Quarter periods
df['Month'] = df['Date'].dt.to_period('M')
df['Quarter'] = df['Date'].dt.to_period('Q')

# Group by Month and Quarter to get total sales
monthly_sales = df.groupby('Month')['Total_Amount'].sum()
quarterly_sales = df.groupby('Quarter')['Total_Amount'].sum()

# Plotting
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# Monthly Trends
# We convert the period index to string so matplotlib plots it nicely
monthly_sales.index = monthly_sales.index.astype(str)
axes[0].plot(monthly_sales.index, monthly_sales.values, marker='o', color='b')
axes[0].set_title('Monthly Sales Trend')
axes[0].set_ylabel('Total Revenue')
axes[0].set_xlabel('Month')
axes[0].tick_params(axis='x', rotation=45)

# Quarterly Trends
quarterly_sales.index = quarterly_sales.index.astype(str)
axes[1].plot(quarterly_sales.index, quarterly_sales.values, marker='s', color='r')
axes[1].set_title('Quarterly Sales Trend')
axes[1].set_ylabel('Total Revenue')
axes[1].set_xlabel('Quarter')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

**Observations:**
- **Monthly Trend:** The line chart shows the peaks and valleys in sales throughout the year. We can identify which specific months generate the most revenue (e.g., holiday seasons, summer).
- **Quarterly Trend:** This provides a smoother, high-level view of business performance over the quarters (Q1, Q2, Q3, Q4). Upward trends signify overall business growth.

## 4. Customer Demographics Analysis
**Checklist Item:** Distribution of customer age groups, gender breakdown

Understanding *who* our customers are is vital for targeted marketing.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age Distribution (Histogram)
sns.histplot(df['Age'], bins=15, kde=True, ax=axes[0], color='purple')
axes[0].set_title('Distribution of Customer Ages')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Number of Customers')

# Gender Breakdown (Pie Chart)
gender_counts = df['Gender'].value_counts()
axes[1].pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%', startangle=90, colors=['#ff9999','#66b3ff'])
axes[1].set_title('Customer Gender Breakdown')

plt.tight_layout()
plt.show()

**Observations:**
- **Age:** The histogram (with a smooth KDE line) reveals our primary customer age bracket. For instance, if the peak is around 25-35, our core audience is young adults.
- **Gender:** The pie chart gives a straightforward percentage breakdown of male vs. female customers, helping to tailor marketing campaigns to the dominant group if a large skew exists.

## 5. Product Analysis
**Checklist Item:** Top 10 best-selling products; revenue by product category (bar chart)

*Note: Since this dataset provides `Product_Category` rather than individual product names, we will analyze the top-selling categories by quantity and the total revenue generated by each category.*

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 12))

# Top categories by quantity sold
top_categories_qty = df.groupby('Product_Category')['Quantity'].sum().sort_values(ascending=False).head(10)
sns.barplot(x=top_categories_qty.values, y=top_categories_qty.index, ax=axes[0], palette='viridis')
axes[0].set_title('Top Product Categories by Quantity Sold')
axes[0].set_xlabel('Total Quantity Sold')
axes[0].set_ylabel('Product Category')

# Revenue by product category
category_revenue = df.groupby('Product_Category')['Total_Amount'].sum().sort_values(ascending=False)
sns.barplot(x=category_revenue.values, y=category_revenue.index, ax=axes[1], palette='magma')
axes[1].set_title('Total Revenue by Product Category')
axes[1].set_xlabel('Total Revenue ($)')
axes[1].set_ylabel('Product Category')

plt.tight_layout()
plt.show()

**Observations:**
- The first chart shows which categories sell the most physical units. This is important for stock and inventory management.
- The second chart shows which categories bring in the most money. Sometimes a category sells fewer units but generates more revenue because the items are more expensive (e.g., Electronics).

## 6. Correlation Heatmap
**Checklist Item:** Heatmap of correlation matrix between numerical variables

A correlation matrix shows how strongly variables are related to each other. Values range from -1 to 1.
- **1** means perfect positive correlation (as one goes up, the other goes up).
- **-1** means perfect negative correlation (as one goes up, the other goes down).
- **0** means no correlation.

In [ ]:
plt.figure(figsize=(10, 8))

# Select only numerical columns for correlation
numerical_df = df.select_dtypes(include=['float64', 'int64', 'int32'])

# Calculate correlation matrix
corr_matrix = numerical_df.corr()

# Plot heatmap
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Heatmap of Numerical Variables')
plt.show()

**Observations:**
- We can see the relationship between `Total_Amount`, `Unit_Price`, and `Quantity`. Naturally, `Total_Amount` is highly correlated with `Unit_Price` and `Quantity`.
- We can check if `Age` correlates with `Total_Amount` (do older customers spend more?). Usually, this correlation is weak (close to 0).
- We can also observe if `Discount_Amount` impacts `Quantity` purchased.

## 7. Additional Visualization: Delivery Time vs. Customer Rating
**Checklist Item:** At least one additional visualization that reveals a non-obvious insight

Let's investigate how the speed of delivery (`Delivery_Time_Days`) affects customer satisfaction (`Customer_Rating`). A boxplot is excellent for showing the distribution of delivery times for each rating out of 5.

In [ ]:
plt.figure(figsize=(10, 6))

# Boxplot comparing Customer Rating and Delivery Time
sns.boxplot(x='Customer_Rating', y='Delivery_Time_Days', data=df, palette='Set2')

plt.title('Impact of Delivery Time on Customer Rating')
plt.xlabel('Customer Rating (1 to 5)')
plt.ylabel('Delivery Time (Days)')
plt.show()

**Observations:**
- This boxplot reveals a potentially non-obvious insight: do 1-star ratings generally correspond to longer delivery times?
- The median line in each box shows the typical delivery time for that rating. If the boxes for lower ratings are higher up on the y-axis, it proves that slow delivery is a major driver of customer dissatisfaction.

## 8. Conclusion & Business Recommendations
Based on our Exploratory Data Analysis, here are 3 actionable business recommendations:

1. **Capitalize on Peak Sales Periods:** The time-series analysis reveals specific months/quarters where sales spike. **Recommendation:** Allocate more marketing budget and ensure inventory is fully stocked 1-2 months prior to these peak seasons to maximize revenue.
2. **Optimize Inventory based on Revenue vs. Volume:** The product analysis showed differences between top-selling items by quantity vs. revenue. **Recommendation:** Keep high-quantity items in stock to drive store traffic and customer acquisition, but heavily promote the high-revenue categories (like Electronics) through targeted upselling to boost the bottom line.
3. **Improve Delivery Logistics to Boost Ratings:** The relationship between delivery days and customer ratings shows that delivery delays harm customer satisfaction. **Recommendation:** Partner with faster logistics providers or offer premium expedited shipping options to improve the overall customer experience and secure returning customers.